# Enhanced Big Data Fertilizer Recommendation with Deep Learning and API Integration

This notebook implements an advanced big data approach using PySpark, deep learning, and real-time weather API integration for fertilizer recommendations.

In [ ]:
# Install required packages
!pip install pyspark pandas tensorflow requests plotly nbformat ipython

Defaulting to user installation because normal site-packages is not writeable
  Using cached tensorflow-2.21.0-cp313-cp313-win_amd64.whl.metadata (4.5 kB)
Using cached tensorflow-2.21.0-cp313-cp313-win_amd64.whl (351.2 MB)


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\gladw\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python313\\site-packages\\tensorflow\\include\\external\\com_github_grpc_grpc\\src\\core\\credentials\\call\\gcp_service_account_identity\\gcp_service_account_identity_credentials.h'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths



In [17]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.classification import MultilayerPerceptronClassifier
import pyspark.sql.functions as F
import requests
import pandas as pd
import numpy as np
from datetime import datetime

# OpenWeatherMap API Key
API_KEY = "90e1198660378e2e03ba71657bdf4849"

# Create Spark session with more resources for deep learning
spark = SparkSession.builder \
    .appName("EnhancedFertilizerRecommendation") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

In [18]:
def get_weather_data(lat, lon):
    """
    Fetch real-time weather data from OpenWeatherMap API.
    """
    url = f"http://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={API_KEY}&units=metric"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        return {
            'temperature': data['main']['temp'],
            'humidity': data['main']['humidity'],
            'pressure': data['main']['pressure']
        }
    else:
        # Return default values if API fails
        return {'temperature': 25, 'humidity': 60, 'pressure': 1013}

# Example usage (you can replace with actual coordinates)
weather = get_weather_data(28.6139, 77.2090)  # Delhi coordinates
print(f"Current weather: {weather}")

Current weather: {'temperature': 34.07, 'humidity': 29, 'pressure': 1008}


In [19]:
# Load and augment data
df = spark.read.csv("Fertilizer Prediction.csv", header=True, inferSchema=True)

# Add weather data (simplified - in real implementation, use actual coordinates)
df = df.withColumn("Weather_Temp", F.lit(weather['temperature']))
df = df.withColumn("Weather_Humidity", F.lit(weather['humidity']))

print(f"Dataset with weather data: {df.count()} rows")
df.show(5)

Dataset with weather data: 100000 rows
+-----------+--------+--------+---------+-----------+--------+---------+-----------+---------------+------------+----------------+
|Temparature|Humidity|Moisture|Soil Type|  Crop Type|Nitrogen|Potassium|Phosphorous|Fertilizer Name|Weather_Temp|Weather_Humidity|
+-----------+--------+--------+---------+-----------+--------+---------+-----------+---------------+------------+----------------+
|         32|      51|      41|      Red|Ground Nuts|       7|        3|         19|       14-35-14|       34.07|              29|
|         35|      58|      35|    Black|     Cotton|       4|       14|         16|           Urea|       34.07|              29|
|         27|      55|      43|    Sandy|  Sugarcane|      28|        0|         17|          20-20|       34.07|              29|
|         33|      56|      56|    Loamy|Ground Nuts|      37|        5|         24|          28-28|       34.07|              29|
|         32|      70|      60|      Red|Gro

In [20]:
# Count unique fertilizer types for neural network output layer
num_classes = df.select("Fertilizer Name").distinct().count()
print(f"Number of fertilizer classes: {num_classes}")

# Index categorical columns
soil_indexer = StringIndexer(inputCol="Soil Type", outputCol="Soil_Type_Index")
crop_indexer = StringIndexer(inputCol="Crop Type", outputCol="Crop_Type_Index")
fert_indexer = StringIndexer(inputCol="Fertilizer Name", outputCol="label")

# Assemble features including weather data
assembler = VectorAssembler(
    inputCols=["Temparature", "Humidity", "Moisture", "Soil_Type_Index", "Crop_Type_Index", 
               "Nitrogen", "Potassium", "Phosphorous", "Weather_Temp", "Weather_Humidity"],
    outputCol="features"
)

# Scale features
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")

# Split data
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

Number of fertilizer classes: 7


In [21]:
# Define layers for neural network (input size, hidden layers, output size)
# 10 input features, hidden layers, and output layer matches number of classes
layers = [10, 20, 15, num_classes]

# Create Multilayer Perceptron Classifier
mlp = MultilayerPerceptronClassifier(
    featuresCol="scaledFeatures",
    labelCol="label",
    layers=layers,
    seed=42,
    maxIter=100
)

# Create pipeline with MLP
pipeline_mlp = Pipeline(stages=[soil_indexer, crop_indexer, fert_indexer, assembler, scaler, mlp])

# Train the model
model_mlp = pipeline_mlp.fit(train_df)

# Also create Random Forest for comparison
rf = RandomForestClassifier(featuresCol="scaledFeatures", labelCol="label", numTrees=100, seed=42)
pipeline_rf = Pipeline(stages=[soil_indexer, crop_indexer, fert_indexer, assembler, scaler, rf])
model_rf = pipeline_rf.fit(train_df)

In [ ]:
# Make predictions with both models
predictions_mlp = model_mlp.transform(test_df)
predictions_rf = model_rf.transform(test_df)

# Evaluate MLP
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy_mlp = evaluator.evaluate(predictions_mlp)
f1_mlp = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1").evaluate(predictions_mlp)

# Evaluate RF
accuracy_rf = evaluator.evaluate(predictions_rf)
f1_rf = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1").evaluate(predictions_rf)

print(f"MLP - Accuracy: {accuracy_mlp:.4f}, F1: {f1_mlp:.4f}")
print(f"RF - Accuracy: {accuracy_rf:.4f}, F1: {f1_rf:.4f}")

# Show some predictions
predictions_mlp.select("Fertilizer Name", "prediction", "probability").show(10)

In [ ]:
# Convert to Pandas for visualization
# First, check if evaluation variables exist; if not, run evaluation
try:
    # Test if variables are defined
    _ = accuracy_mlp
except NameError:
    # If not defined, run evaluation
    print("Evaluation variables not found. Running evaluation...")
    
    predictions_mlp = model_mlp.transform(test_df)
    predictions_rf = model_rf.transform(test_df)
    
    evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
    accuracy_mlp = evaluator.evaluate(predictions_mlp)
    f1_mlp = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1").evaluate(predictions_mlp)
    
    accuracy_rf = evaluator.evaluate(predictions_rf)
    f1_rf = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1").evaluate(predictions_rf)
    
    print(f"MLP - Accuracy: {accuracy_mlp:.4f}, F1: {f1_mlp:.4f}")
    print(f"RF - Accuracy: {accuracy_rf:.4f}, F1: {f1_rf:.4f}")

# Create results dataframe for visualization
results_df = pd.DataFrame({
    'Model': ['MLP', 'Random Forest'],
    'Accuracy': [accuracy_mlp, accuracy_rf],
    'F1_Score': [f1_mlp, f1_rf]
})

print("\nModel Comparison Results:")
print(results_df)

# Create visualizations using both Plotly and Matplotlib
import plotly.express as px
import matplotlib.pyplot as plt

# Plotly bar chart - save as HTML
fig = px.bar(results_df, x='Model', y='Accuracy', title='Model Accuracy Comparison', 
             labels={'Accuracy': 'Accuracy Score'}, color='Model')
fig.write_html('model_accuracy_comparison.html')
print("\nPlotly visualization saved to model_accuracy_comparison.html")

# Create matplotlib figure for display
fig_mpl, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy comparison
ax1.bar(results_df['Model'], results_df['Accuracy'], color=['#636EFA', '#EF553B'])
ax1.set_ylabel('Accuracy Score')
ax1.set_title('Model Accuracy Comparison')
ax1.set_ylim([0, 1])
for i, v in enumerate(results_df['Accuracy']):
    ax1.text(i, v + 0.02, f'{v:.4f}', ha='center')

# F1 Score comparison
ax2.bar(results_df['Model'], results_df['F1_Score'], color=['#636EFA', '#EF553B'])
ax2.set_ylabel('F1 Score')
ax2.set_title('Model F1 Score Comparison')
ax2.set_ylim([0, 1])
for i, v in enumerate(results_df['F1_Score']):
    ax2.text(i, v + 0.02, f'{v:.4f}', ha='center')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
print("Matplotlib visualization saved to model_comparison.png")
plt.show()

# Save results
results_df.to_csv('model_comparison_results.csv', index=False)
print("Results saved to model_comparison_results.csv")

spark.stop()

Evaluation variables not found. Running evaluation...
MLP - Accuracy: 0.1424, F1: 0.1081
RF - Accuracy: 0.1397, F1: 0.1125

Model Comparison Results:
           Model  Accuracy  F1_Score
0            MLP  0.142395  0.108129
1  Random Forest  0.139659  0.112503


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed